# B3AC — Area cap (C) and part filter on the Small base (P on [S])

## Why
- **Area cap.** GPV_K-n+P[MixA] is the best overall system (reweighted 24.38, −4.27 vs H1b_v1) but fails T1 (S1 10.68): in sparse images K3n splits large single objects (B3PN: the normals merge detector did not help — large single objects have internal normal boundaries). UM showed that area ÷ object size is a poor **estimator** (it amplifies errors) but it is a natural **upper bound**: a region cannot hold more objects than its area allows.
- **P on [S].** The part filter improved every base it was applied to (B3PN). If it gives most of the gain on the existing SAM2.1-Small base, a candidate would need almost no new held-out caching.

## Pre-declared design (before any B3AC result)
| Item | Rule |
|---|---|
| Object size s | median area of the mask set's verified single masks (target-dominant in the tiled expanded-vocabulary map, not rule-A merged, SAM2 score ≥ median); fallbacks target-dominant, all (B3PN definition), computed on the mask set being counted |
| **C — area cap** | in GPV_K-n and H1b-K3n, each merged region's multiplicity = **min( max(1, K3n segments), max(1, round(region area / s)) )** |
| **P** | B3PN part filter (area < 0.25 s and > 80% inside another kept mask) |
| Bases | **MixA**, **LM0**, **S** (SAM2.1-Small, full image: frozen caches) |
| Variants per base | GPV_R4, GPV_R4+P, GPV_K-n, GPV_K-n+P, GPV_K-n+C, GPV_K-n+P+C, H1b, H1b+P, H1b-K3n+P, H1b-K3n+P+C |
| Targets | T1, T2′, T3′ |

Reproduction check: MixA and LM0 GPV_R4, GPV_R4+P, GPV_K-n, GPV_K-n+P must equal B3PN.

## Run order
CPU → 1.0–1.3 → 2.1 → 3.1 (resumable) → 4.1.

# 1. Setup

In [1]:
# 1.0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 1.1 — Paths, repo modules, train pool, frozen-system counts
# ============================================================

import importlib, json, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
FSC147_RAW = PROJECT_ROOT / "datasets" / "FSC147" / "raw"
FSC147_IMAGE_DIR = FSC147_RAW / "images_384_VarV2"
TD_ROOT = PROJECT_ROOT / "cache" / "FSC147_train_dev400"
B3S_DIR = RESULTS_ROOT / "B3S_Semantic_Improvement"
OUT_DIR = RESULTS_ROOT / "B3AC_Area_Cap_and_P_on_S"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import src.pipeline.io_cache as io_cache
import src.pipeline.sg as sg
import src.pipeline.gpv as gpv
import src.pipeline.hybrid as hybrid
import src.pipeline.b0 as b0
for m in (io_cache, sg, gpv, hybrid, b0):
    importlib.reload(m)

STRATA = ["S1_0_20", "S2_20_40", "S3_40_60", "S4_60_80", "S5_80_95", "S6_95_100"]
W = {"S1_0_20": .20, "S2_20_40": .20, "S3_40_60": .20, "S4_60_80": .20, "S5_80_95": .15, "S6_95_100": .05}
TD_DIRS = {"b0_masks": TD_ROOT / "b0_masks", "a3c_masks": TD_ROOT / "a3c_masks",
           "depth": TD_ROOT / "depth", "m1_split_labels": TD_ROOT / "m1_split_labels"}
SG_CFG = sg.SGConfig.from_frozen_json(RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "SG_v1.1_frozen_config.json")

td_df = pd.read_csv(TD_ROOT / "FSC147_train_dev400.csv")
td_res = pd.read_csv(B3S_DIR / "B3S_step1_per_image.csv")
assert len(td_df) == 400 and len(td_res) == 400
print("Repo commit:", REPO_COMMIT, "| output:", OUT_DIR)

Repo commit: d70bb1d6d631c12156ba7bf1881d22f5b2eb6100 | output: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3AC_Area_Cap_and_P_on_S


In [3]:
import torch

# ============================================================
# Step 1.2 — Shared helpers: guided masks, B0 meta filtering, systems, targets
# ============================================================

def guided_masks_for_prompts(predictor, image_np, prompts_xy):
    h, w = image_np.shape[:2]
    masks, scores = [], []
    if len(prompts_xy):
        with torch.inference_mode():
            predictor.set_image(image_np)
            for p in prompts_xy:
                m, s, _ = predictor.predict(point_coords=np.asarray([p], dtype=np.float32),
                                            point_labels=np.asarray([1], dtype=np.int32),
                                            multimask_output=False)
                masks.append(np.asarray(m[0], dtype=bool)); scores.append(float(np.asarray(s)[0]))
    return (np.asarray(masks) if masks else np.zeros((0, h, w), bool)), np.asarray(scores, float)

def filter_meta(meta, keep):
    return {k: np.asarray(v)[keep] for k, v in meta.items()}

def systems_from_masks(b0_masks, b0_iou, g_masks, g_scores, a3c, depth, san_target):
    keep = gpv.nms_masks(g_masks, g_scores, 0.5) if g_masks.shape[0] else np.zeros(0, int)
    r4_m, _, _ = gpv.build_r4(g_masks[keep], np.asarray(g_scores)[keep], b0_masks, b0_iou, 0.5)
    d1 = gpv.d1_peaks(san_target)
    h1b = float(hybrid.h1b_count(r4_m, d1, a3c, depth, SG_CFG)[0])
    return {"B0": int(b0_masks.shape[0]), "GPV_R4": int(r4_m.shape[0]), "H1b": h1b}

def evaluate(frame, systems, label):
    gt = frame["gt_count"].to_numpy(float); st = frame["density_stratum"].to_numpy()
    AE = {s: np.abs(frame[s].to_numpy(float) - gt) for s in systems}
    IDX = {}
    for j, k in enumerate(STRATA):
        pos = np.flatnonzero(st == k)
        IDX[k] = pos[np.random.default_rng(1000 + j).integers(0, len(pos), size=(10_000, len(pos)))]
    def sci(a, b, k):
        d = AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)
        return AE[a][st == k].mean() - AE[b][st == k].mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)
    def rci(a, b):
        d = sum(w * (AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)) for k, w in W.items())
        o = sum(w * (AE[a][st == k].mean() - AE[b][st == k].mean()) for k, w in W.items())
        return o, np.percentile(d, 2.5), np.percentile(d, 97.5)
    res = pd.DataFrame([{"system": s,
                         "MAE_reweighted": sum(w * AE[s][st == k].mean() for k, w in W.items()),
                         "mean_signed": float((frame[s] - frame["gt_count"]).mean()),
                         **{f"MAE_{k[:2]}": AE[s][st == k].mean() for k in STRATA}} for s in systems])
    rows = []
    for s in [x for x in systems if x not in ("B0_v1", "GPV_R4_v1", "H1b_v1")]:
        rec = {"system": s}; t1 = []; t2 = []
        for k in ["S1_0_20", "S2_20_40"]:
            d, lo, hi = sci(s, "B0_v1", k); rec[f"{k[:2]} vs B0"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t1.append(lo <= 0)
        for k in ["S5_80_95", "S6_95_100"]:
            d, lo, hi = sci(s, "H1b_v1", k); rec[f"{k[:2]} vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t2.append(lo <= 0)
        d, lo, hi = rci(s, "H1b_v1"); rec["overall vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"
        rec.update({"T1": all(t1), "T2'": all(t2), "T3'": hi < 0, "ALL": all(t1) and all(t2) and hi < 0})
        rows.append(rec)
    print(f"{label} — metrics:"); display(res.round(2))
    print("Targets (difference [95% CI]; negative = system better):"); display(pd.DataFrame(rows))
    return res, pd.DataFrame(rows)
print("helpers ready")

helpers ready


In [4]:
# ============================================================
# Step 1.3 — Normals edge map and K3n counting (B3MN definitions)
# ============================================================

from scipy import ndimage
import src.pipeline.multiplicity as K
importlib.reload(K)
N_ROOT = TD_ROOT / "normals"

def normal_edge_map(nrm):
    n = nrm.astype(np.float32)
    n = n / np.maximum(np.linalg.norm(n, axis=2, keepdims=True), 1e-6)
    edge = np.zeros(n.shape[:2], np.float32)
    for dy, dx in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
        edge = np.maximum(edge, 1.0 - (n * np.roll(n, shift=(dy, dx), axis=(0, 1))).sum(axis=2))
    edge[0, :] = edge[1, :]; edge[-1, :] = edge[-2, :]; edge[:, 0] = edge[:, 1]; edge[:, -1] = edge[:, -2]
    return ndimage.gaussian_filter(edge, 1.0)

def k3n_segments(region, edge, min_seg, h_frac=0.1):
    e = edge[region]
    h = h_frac * float(np.percentile(e, 95) - np.percentile(e, 5))
    return 1 if h <= 0 else K._marker_watershed(edge, region, -edge, h, min_seg)

def gpv_k_n_count(r4_m, r4_s, d1, edge):
    masks = np.asarray(r4_m, bool)
    if masks.shape[0] == 0:
        return 0
    flagged = gpv.merged_mask_flags(masks, gpv.points_per_mask(masks, d1), "A")
    a_ref = K.reference_area(masks, flagged)
    n_nonflag = int((~flagged).sum())
    regions = [rg for _, rg in K.build_regions(masks, r4_s, flagged) if rg.sum() > 0 and rg.sum() >= 0.5 * a_ref]
    if not (regions and sum(K.k1_peaks(rg, d1) for rg in regions) > n_nonflag):
        return int(masks.shape[0])
    return n_nonflag + sum(max(1, k3n_segments(rg, edge, 0.25 * a_ref)) for rg in regions)

def h1b_k3n_count(r4_m, d1, a3c, depth, edge):
    base, info = hybrid.h1b_count(r4_m, d1, a3c, depth, SG_CFG)
    if not info["stage2_on"] or r4_m.shape[0] == 0:
        return base
    merged = gpv.merged_mask_flags(r4_m, gpv.points_per_mask(r4_m, d1), "A")
    if not merged.any():
        return base
    anchor = r4_m[~merged]
    areas = anchor.reshape(anchor.shape[0], -1).sum(1) if anchor.shape[0] else np.zeros(0)
    a_ref = float(np.median(areas)) if areas.size else float(np.median(r4_m[merged].reshape(merged.sum(), -1).sum(1)))
    taken = anchor.any(axis=0) if anchor.shape[0] else np.zeros(r4_m.shape[1:], bool)
    total = int(anchor.shape[0])
    for i in np.flatnonzero(merged):
        rg = r4_m[i] & ~taken
        taken |= rg
        if rg.sum() > 0 and rg.sum() >= 0.5 * a_ref:
            total += max(1, k3n_segments(rg, edge, 0.25 * a_ref))
    return float(total)

def all_counts(masks, scores, d1, a3c, depth, edge):
    return {"GPV_R4": int(masks.shape[0]),
            "H1b": float(hybrid.h1b_count(masks, d1, a3c, depth, SG_CFG)[0]),
            "GPV_K-n": gpv_k_n_count(masks, scores, d1, edge),
            "H1b-K3n": h1b_k3n_count(masks, d1, a3c, depth, edge)}
print("K3n counting ready")

K3n counting ready


# 2. Counting with and without the cap

In [5]:
# ============================================================
# Step 2.1 — Object size, P, and capped / uncapped K3n counting
# ============================================================

TD_TILED = TD_ROOT / "san_vocab_tiled"
PART_FRAC, NEST_FRAC = 0.25, 0.80

def object_size(masks, scores, amax, d1):
    n = masks.shape[0]
    areas = masks.reshape(n, -1).sum(1).astype(float)
    n_cls = int(amax.max()) + 1
    dom_t = np.array([np.bincount(amax[m], minlength=n_cls).argmax() == 0 if m.any() else False for m in masks])
    merged = gpv.merged_mask_flags(masks, gpv.points_per_mask(masks, d1), "A")
    verified = dom_t & ~merged & (np.asarray(scores) >= np.median(scores))
    if verified.sum() >= 3:
        return float(np.median(areas[verified]))
    if dom_t.any():
        return float(np.median(areas[dom_t]))
    return float(np.median(areas))

def part_filter(masks, scores, s):
    n = masks.shape[0]
    if n < 2:
        return masks, np.asarray(scores, float)
    flat = masks.reshape(n, -1).astype(np.float32)
    areas = flat.sum(1)
    inter = flat @ flat.T
    np.fill_diagonal(inter, 0)
    drop = (areas < PART_FRAC * s) & (inter.max(axis=1) > NEST_FRAC * np.maximum(areas, 1))
    return masks[~drop], np.asarray(scores, float)[~drop]

def cap(k, area, s):
    return min(max(1, k), max(1, int(round(area / max(s, 1.0)))))

def gpv_k_n_both(masks, scores, d1, edge, s):
    # returns (uncapped, capped) GPV_K-n counts
    masks = np.asarray(masks, bool)
    if masks.shape[0] == 0:
        return 0, 0
    flagged = gpv.merged_mask_flags(masks, gpv.points_per_mask(masks, d1), "A")
    a_ref = K.reference_area(masks, flagged)
    n_nonflag = int((~flagged).sum())
    regions = [rg for _, rg in K.build_regions(masks, scores, flagged) if rg.sum() > 0 and rg.sum() >= 0.5 * a_ref]
    if not (regions and sum(K.k1_peaks(rg, d1) for rg in regions) > n_nonflag):
        return int(masks.shape[0]), int(masks.shape[0])
    segs = [max(1, k3n_segments(rg, edge, 0.25 * a_ref)) for rg in regions]
    return n_nonflag + sum(segs), n_nonflag + sum(cap(k, rg.sum(), s) for k, rg in zip(segs, regions))

def h1b_k3n_both(masks, d1, a3c, depth, edge, s):
    base, info = hybrid.h1b_count(masks, d1, a3c, depth, SG_CFG)
    if not info["stage2_on"] or masks.shape[0] == 0:
        return base, base, base
    merged = gpv.merged_mask_flags(masks, gpv.points_per_mask(masks, d1), "A")
    if not merged.any():
        return base, base, base
    anchor = masks[~merged]
    areas = anchor.reshape(anchor.shape[0], -1).sum(1) if anchor.shape[0] else np.zeros(0)
    a_ref = float(np.median(areas)) if areas.size else float(np.median(masks[merged].reshape(merged.sum(), -1).sum(1)))
    taken = anchor.any(axis=0) if anchor.shape[0] else np.zeros(masks.shape[1:], bool)
    unc = cpd = int(anchor.shape[0])
    for i in np.flatnonzero(merged):
        rg = masks[i] & ~taken
        taken |= rg
        if rg.sum() > 0 and rg.sum() >= 0.5 * a_ref:
            k = max(1, k3n_segments(rg, edge, 0.25 * a_ref))
            unc += k; cpd += cap(k, rg.sum(), s)
    return base, float(unc), float(cpd)      # (H1b, H1b-K3n, H1b-K3n+C)
print("Capped / uncapped counting ready")

Capped / uncapped counting ready


# 3. All variants on MixA, LM0 and S

In [6]:
# ============================================================
# Step 3.1 — Variants on the three bases (resumable per image)
# ============================================================

LM_ROOT = TD_ROOT / "sam2_large_masked_M0"
ZLM_ROOT = TD_ROOT / "zoomed" / "LM0"
ROWS = OUT_DIR / "rows"
ROWS.mkdir(parents=True, exist_ok=True)

def r4_large(b0_dir, guided_dir, di, stem):
    bm, bmeta = io_cache.load_b0_masks(b0_dir / "b0" / f"{int(di):05d}_{stem}.npz")
    g = io_cache.load_guided_masks(guided_dir / "guided" / f"{int(di):04d}_{stem}.npz")
    kg = gpv.nms_masks(g["masks"], g["scores"], 0.5) if g["masks"].shape[0] else np.zeros(0, int)
    m, s, _ = gpv.build_r4(g["masks"][kg], np.asarray(g["scores"])[kg], bm, bmeta["predicted_iou"], 0.5)
    return m, np.asarray(s, float)

def r4_small(di, stem, p):
    bm, bmeta = io_cache.load_b0_masks(p["b0_masks"])
    g = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery" / f"{int(di):04d}_{stem}.npz")
    kg = gpv.nms_masks(g["masks"], g["scores"], 0.5)
    m, s, _ = gpv.build_r4(g["masks"][kg], g["scores"][kg], bm, bmeta["predicted_iou"], 0.5)
    return m, np.asarray(s, float)

todo = [r for r in td_df.itertuples(index=False) if not (ROWS / f"{int(r.dataset_index):05d}.json").exists()]
print(f"Done: {len(td_df) - len(todo)} | to process: {len(todo)}")
t0 = time.time()
for r in tqdm(todo, desc="Cap / P variants"):
    di, stem = r.dataset_index, io_cache.image_stem(r.image_id)
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    dep = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    d1 = gpv.d1_peaks(san["target_score"])
    with np.load(N_ROOT / f"{stem}.npz") as v:
        edge = normal_edge_map(v["normals"])
    with np.load(TD_TILED / f"{stem}.npz") as v:
        amax = v["tiled_argmax"]
    row = {"dataset_index": int(di)}
    sets = {"MixA": r4_large(ZLM_ROOT, LM_ROOT, di, stem), "LM0": r4_large(LM_ROOT, LM_ROOT, di, stem),
            "S": r4_small(di, stem, p)}
    for b, (m, sc) in sets.items():
        if m.shape[0] == 0:
            row.update({f"{v}[{b}]": 0 for v in ["GPV_R4", "GPV_R4+P", "GPV_K-n", "GPV_K-n+P", "GPV_K-n+C",
                                                   "GPV_K-n+P+C", "H1b", "H1b+P", "H1b-K3n+P", "H1b-K3n+P+C"]})
            continue
        s_full = object_size(m, sc, amax, d1)
        mp, sp = part_filter(m, sc, s_full)
        s_p = object_size(mp, sp, amax, d1) if mp.shape[0] else s_full
        k_unc, k_cap = gpv_k_n_both(m, sc, d1, edge, s_full)
        kp_unc, kp_cap = gpv_k_n_both(mp, sp, d1, edge, s_p)
        h_base = float(hybrid.h1b_count(m, d1, a3c, dep, SG_CFG)[0])
        hp_base, hp_unc, hp_cap = h1b_k3n_both(mp, d1, a3c, dep, edge, s_p)
        row.update({f"GPV_R4[{b}]": int(m.shape[0]), f"GPV_R4+P[{b}]": int(mp.shape[0]),
                    f"GPV_K-n[{b}]": k_unc, f"GPV_K-n+P[{b}]": kp_unc,
                    f"GPV_K-n+C[{b}]": k_cap, f"GPV_K-n+P+C[{b}]": kp_cap,
                    f"H1b[{b}]": h_base, f"H1b+P[{b}]": hp_base,
                    f"H1b-K3n+P[{b}]": hp_unc, f"H1b-K3n+P+C[{b}]": hp_cap,
                    f"object_size[{b}]": s_full})
    with open(ROWS / f"{int(di):05d}.json", "w") as f:
        json.dump(row, f)
print(f"Elapsed: {(time.time() - t0) / 60:.1f} min")

Done: 0 | to process: 400


Cap / P variants:   0%|          | 0/400 [00:00<?, ?it/s]

Elapsed: 43.1 min


# 4. Results and targets

In [6]:
# ============================================================
# Step 4.1 — Reproduction check, metrics and targets
#            (rows copied to local disk first: Drive reads of many small files can stall)
# ============================================================

import shutil
LOCAL_ROWS = Path("/content/rows_local")
if LOCAL_ROWS.exists():
    shutil.rmtree(LOCAL_ROWS)
t0 = time.time()
shutil.copytree(OUT_DIR / "rows", LOCAL_ROWS)
print(f"Copied {len(list(LOCAL_ROWS.glob('*.json')))} files in {time.time() - t0:.0f} s")

ac = pd.DataFrame([json.load(open(p)) for p in sorted(LOCAL_ROWS.glob("*.json"))])
assert len(ac) == 400, len(ac)

pn = pd.read_csv(RESULTS_ROOT / "B3PN_Part_and_Normals_Merge" / "B3PN_per_image.csv")
chk = ac.merge(pn, on="dataset_index", suffixes=("", "_pn"))
print("Reproduction vs B3PN (images equal / 400):")
for c in ["GPV_R4[MixA]", "GPV_R4+P[MixA]", "GPV_K-n[MixA]", "GPV_K-n+P[MixA]",
          "GPV_R4[LM0]", "GPV_R4+P[LM0]", "GPV_K-n[LM0]", "GPV_K-n+P[LM0]"]:
    print(f"  {c:<18}: {int(np.isclose(chk[c], chk[c + '_pn']).sum())}")
ref = ac.merge(td_res, on="dataset_index")
print("Small base reproduces frozen counts — GPV_R4[S] vs GPV_R4_v1:",
      int((ref["GPV_R4[S]"] == ref["GPV_R4_v1"]).sum()),
      "| H1b[S] vs H1b_v1:", int(np.isclose(ref["H1b[S]"], ref["H1b_v1"]).sum()))

frame = ac.merge(td_res[["dataset_index", "gt_count", "density_stratum", "B0", "GPV_R4_v1", "H1b_v1"]]
                 .rename(columns={"B0": "B0_v1"}), on="dataset_index", validate="one_to_one")
VARS = ["GPV_R4", "GPV_R4+P", "GPV_K-n", "GPV_K-n+P", "GPV_K-n+C", "GPV_K-n+P+C", "H1b+P", "H1b-K3n+P", "H1b-K3n+P+C"]
SYS = ["B0_v1", "GPV_R4_v1", "H1b_v1"] + [f"{v}[{b}]" for b in ["MixA", "LM0", "S"] for v in VARS]
t0 = time.time()
res, targets = evaluate(frame, SYS, "Area cap and P on the Small base (train_dev400)")
print(f"Evaluation: {time.time() - t0:.0f} s")

frame.to_csv(OUT_DIR / "B3AC_per_image.csv", index=False)
res.to_csv(OUT_DIR / "B3AC_metrics.csv", index=False)
targets.to_csv(OUT_DIR / "B3AC_targets.csv", index=False)
with open(OUT_DIR / "B3AC_metadata.json", "w") as f:
    json.dump({"notebook": "B3AC_Area_Cap_and_P_on_S",
               "cap": "region multiplicity = min(max(1, K3n), max(1, round(region area / object size)))",
               "object_size": "median area of verified singles on the counted mask set (B3PN definition)",
               "P": f"area < {PART_FRAC} x object size AND > {NEST_FRAC:.0%} inside another kept mask",
               "bases": {"MixA": "B0 Z-LM0 + guided LM0", "LM0": "Large + masked M0", "S": "Small, full image (frozen)"},
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", OUT_DIR)

Copied 400 files in 8 s
Reproduction vs B3PN (images equal / 400):
  GPV_R4[MixA]      : 400
  GPV_R4+P[MixA]    : 400
  GPV_K-n[MixA]     : 400
  GPV_K-n+P[MixA]   : 400
  GPV_R4[LM0]       : 400
  GPV_R4+P[LM0]     : 400
  GPV_K-n[LM0]      : 400
  GPV_K-n+P[LM0]    : 400
Small base reproduces frozen counts — GPV_R4[S] vs GPV_R4_v1: 400 | H1b[S] vs H1b_v1: 400
Area cap and P on the Small base (train_dev400) — metrics:


,system,MAE_reweighted,mean_signed,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0_v1,35.67,-62.67,5.56,7.74,13.30,22.93,59.03,338.08
1,GPV_R4_v1,29.94,-46.78,8.62,7.70,12.45,16.13,40.52,297.63
2,H1b_v1,28.65,-41.28,10.46,9.49,12.35,14.20,37.33,274.98
3,GPV_R4[MixA],27.02,-39.64,7.98,7.29,12.12,15.67,37.47,255.92
4,GPV_R4+P[MixA],25.54,-41.96,6.26,5.40,9.73,13.25,37.93,258.37
5,GPV_K-n[MixA],26.11,-32.99,13.02,8.22,12.18,16.02,29.88,234.73
6,GPV_K-n+P[MixA],24.38,-35.73,10.68,5.86,9.18,13.72,30.88,237.18
7,GPV_K-n+C[MixA],26.29,-33.39,12.80,8.15,12.15,16.30,30.85,235.62
8,GPV_K-n+P+C[MixA],24.53,-36.08,10.51,5.84,9.18,13.97,31.37,238.48
9,H1b+P[MixA],25.08,-37.97,8.00,6.42,10.82,13.27,33.25,247.80


Targets (difference [95% CI]; negative = system better):


,system,S1 vs B0,S2 vs B0,S5 vs H1b,S6 vs H1b,overall vs H1b,T1,T2',T3',ALL
0,GPV_R4[MixA],"+2.41 [+0.54, +4.40]","-0.45 [-2.44, +1.64]","+0.13 [-7.00, +8.73]","-19.07 [-36.48, -0.63]","-1.62 [-3.31, +0.15]",False,True,False,False
1,GPV_R4+P[MixA],"+0.70 [-0.99, +2.43]","-2.34 [-4.06, -0.46]","+0.60 [-6.48, +9.23]","-16.62 [-33.90, +1.67]","-3.11 [-4.81, -1.36]",True,True,True,True
2,GPV_K-n[MixA],"+7.46 [+3.78, +11.47]","+0.49 [-1.89, +3.08]","-7.45 [-16.20, +2.42]","-40.25 [-65.99, -15.67]","-2.54 [-4.76, -0.26]",False,True,True,False
3,GPV_K-n+P[MixA],"+5.11 [+1.89, +8.65]","-1.88 [-3.79, +0.16]","-6.45 [-15.05, +3.23]","-37.80 [-63.62, -13.47]","-4.27 [-6.40, -2.06]",False,True,True,False
4,GPV_K-n+C[MixA],"+7.24 [+3.65, +11.14]","+0.41 [-1.93, +2.96]","-6.48 [-14.83, +3.08]","-39.37 [-64.72, -15.23]","-2.36 [-4.52, -0.14]",False,True,True,False
5,GPV_K-n+P+C[MixA],"+4.95 [+1.79, +8.38]","-1.90 [-3.80, +0.14]","-5.97 [-14.28, +3.52]","-36.50 [-61.43, -12.82]","-4.12 [-6.21, -1.95]",False,True,True,False
6,H1b+P[MixA],"+2.44 [+0.34, +4.62]","-1.31 [-3.25, +0.74]","-4.08 [-9.85, +2.95]","-27.18 [-45.67, -7.62]","-3.57 [-5.16, -1.90]",False,True,True,False
7,H1b-K3n+P[MixA],"+2.53 [+0.40, +4.68]","-1.27 [-3.25, +0.80]","-4.02 [-9.97, +3.12]","-30.70 [-51.47, -9.27]","-3.25 [-5.10, -1.31]",False,True,True,False
8,H1b-K3n+P+C[MixA],"+2.53 [+0.40, +4.68]","-1.27 [-3.25, +0.80]","-4.27 [-10.10, +2.77]","-31.35 [-52.32, -9.95]","-3.40 [-5.17, -1.55]",False,True,True,False
9,GPV_R4[LM0],"-0.21 [-1.59, +1.11]","-2.46 [-3.99, -0.89]","+9.95 [+3.22, +18.33]","+42.72 [+29.13, +57.53]","+1.71 [+0.17, +3.37]",True,False,False,False


Evaluation: 1 s
Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3AC_Area_Cap_and_P_on_S
